# Fotos femininas do Diário de Treino

**Como usar:** toque em **Ambiente de execução → Executar tudo** (no celular, fica no menu ☰). Quando pedir, permita o acesso ao seu Google Drive.

Na primeira vez ele faz só o **teste do agachamento** e mostra as fotos no fim da página.

As fotos e o registro ficam na pasta `diario-fotos-femininas` do seu Drive. Se o Colab desligar, é só executar de novo: ele continua de onde parou.


In [ ]:
#@title Escolha o que fazer
FAZER_TODOS = False #@param {type:"boolean"}


In [ ]:
#@title Preparar (demora alguns minutos)
import os, sys, traceback, datetime
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from google.colab import drive
drive.mount("/content/drive")
PASTA = "/content/drive/MyDrive/diario-fotos-femininas"
os.makedirs(PASTA, exist_ok=True)
def registra(msg):
    linha = datetime.datetime.utcnow().strftime("%d/%m %H:%M") + " " + str(msg)
    print(linha)
    with open(PASTA + "/registro.txt", "a", encoding="utf8") as f:
        f.write(linha + "\n")
registra("Inicio. FAZER_TODOS=" + str(FAZER_TODOS))
!pip install -q -U git+https://github.com/huggingface/diffusers peft bitsandbytes accelerate transformers


In [ ]:
#@title Carregar a IA
import math, gc, time, json, io, urllib.request
import torch
from PIL import Image
from IPython.display import display
try:
    if not torch.cuda.is_available():
        raise RuntimeError("Sem GPU: troque em Ambiente de execucao > Alterar o tipo > T4 GPU")
    registra("GPU: " + torch.cuda.get_device_name(0) + " memoria GB: " + str(round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)))
    from diffusers import QwenImageEditPipeline, FlowMatchEulerDiscreteScheduler
    # Versao compacta com licenca Apache 2.0 (uso comercial permitido)
    pipe = QwenImageEditPipeline.from_pretrained("ovedrive/qwen-image-edit-4bit", torch_dtype=torch.bfloat16)
    PASSOS, FORCA = 20, 4.0
    try:
        pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config({
            "base_image_seq_len": 256, "base_shift": math.log(3), "invert_sigmas": False,
            "max_image_seq_len": 8192, "max_shift": math.log(3), "num_train_timesteps": 1000,
            "shift": 1.0, "shift_terminal": None, "stochastic_sampling": False,
            "time_shift_type": "exponential", "use_beta_sigmas": False, "use_dynamic_shifting": True,
            "use_exponential_sigmas": False, "use_karras_sigmas": False})
        pipe.load_lora_weights("lightx2v/Qwen-Image-Lightning", weight_name="Qwen-Image-Edit-Lightning-8steps-V1.0-bf16.safetensors")
        PASSOS, FORCA = 8, 1.0
        registra("Modo rapido ligado")
    except Exception as e:
        registra("Modo rapido indisponivel, seguindo no normal: " + repr(e)[:300])
        pipe = QwenImageEditPipeline.from_pretrained("ovedrive/qwen-image-edit-4bit", torch_dtype=torch.bfloat16)
    pipe.enable_model_cpu_offload()
    registra("IA carregada")
except Exception:
    registra("ERRO AO CARREGAR:\n" + traceback.format_exc()[-1800:])
    raise


In [ ]:
#@title Gerar as fotos
BASE = "https://cdn.jsdelivr.net/gh/yuhonas/free-exercise-db@main/exercises/"
SITE = "https://cleytonarthur-debug.github.io/diario-de-treino/metodos/imagens.json"
PROMPT = ("Replace the man with an adult woman athlete of about 30, fit and natural build, brown hair in a ponytail, "
          "wearing a black fitted sports tank top, black gym leggings and training shoes. "
          "Keep exactly the same body position, stance, equipment, weights, hand grip, camera angle, framing, "
          "gym background and lighting. Realistic gym photograph, correct anatomy.")
def baixa(url):
    pedido = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return Image.open(io.BytesIO(urllib.request.urlopen(pedido, timeout=60).read())).convert("RGB")
def gera(caminho):
    destino = PASTA + "/" + caminho
    if os.path.exists(destino):
        return False
    original = baixa(BASE + caminho)
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    t = time.time()
    with torch.inference_mode():
        foto = pipe(image=original, prompt=PROMPT, negative_prompt=" ", true_cfg_scale=FORCA,
                    num_inference_steps=PASSOS, width=848, height=560,
                    generator=torch.manual_seed(7)).images[0]
    foto.resize(original.size, Image.LANCZOS).save(destino, quality=80)
    registra("Pronta " + caminho + " em " + str(round(time.time() - t)) + " s")
    gc.collect(); torch.cuda.empty_cache()
    return True
TESTE = ["Barbell_Full_Squat/0.jpg", "Barbell_Full_Squat/1.jpg"]
if FAZER_TODOS:
    regras = json.loads(urllib.request.urlopen(SITE, timeout=60).read().decode("utf8"))["regras"]
    lista = list(dict.fromkeys(i for r in regras for i in r["img"]))
else:
    lista = TESTE
registra("Lista com " + str(len(lista)) + " fotos")
erros = 0
for caminho in lista:
    try:
        gera(caminho)
    except Exception:
        erros += 1
        registra("ERRO em " + caminho + ":\n" + traceback.format_exc()[-1200:])
        gc.collect(); torch.cuda.empty_cache()
        if erros >= 5:
            registra("Parei depois de 5 erros")
            break
prontas = sum(os.path.exists(PASTA + "/" + c) for c in lista)
registra("FIM: " + str(prontas) + " de " + str(len(lista)) + " prontas, " + str(erros) + " erros")
for c in TESTE:
    if os.path.exists(PASTA + "/" + c):
        im = Image.open(PASTA + "/" + c); im.thumbnail((640, 640)); display(im)
drive.flush_and_unmount()
